# Agricultural Growth Stage with openEO

This notebook converts the Sentinel Hub
[Agricultural Growth Stage](https://custom-scripts.sentinel-hub.com/custom-scripts/sentinel-2/agriculture_growth_stage/)
evalscript by **@HarelDan** (adapted for quarterly mosaics by **András Zlinszky**)
into a reusable openEO User-Defined Process (UDP).

## Overview

In this notebook, we will:
1. Connect to an openEO backend service
2. Define an agricultural area of interest and a **three-month** growing-season window
3. Load Sentinel-2 L2A imagery (with the Scene Classification Layer)
4. Compute a **mean NDVI for each of the three calendar months** in the window
5. Assign the three monthly means to the **red, green and blue** channels and export
   the result as a reusable openEO process

## What the growth-stage composite does

The original evalscript walks every Sentinel-2 acquisition of the last ~3 months
(`mosaicking: "ORBIT"` + a `preProcessScenes` 3-month filter) and, for each output
pixel, computes the **average NDVI of each month**. The three monthly averages are
stretched from `[0.1, 0.7]` to `[0, 1]` and written to the R, G and B bands
(oldest month → R, middle month → G, most recent month → B).

Because colour encodes **when** in the season a pixel was greenest, crops with
different phenology separate visually:

- A single sharp NDVI peak in one month → a **primary colour** (red / green / blue).
- A long, even growing season → **yellow** (R+G) or **cyan** (G+B).
- A double peak with a dry gap in between → **purple/magenta** (R+B).
- Bare soil / water (consistently low NDVI) → **dark**.

### How the conversion maps to openEO

| Original (evalscript) | openEO conversion |
| --- | --- |
| `mosaicking: "ORBIT"` + 3-month `preProcessScenes` filter | three-month `temporal_extent` on `load_collection` |
| `calcNDVI(sample)` per orbit | `(B08 - B04) / (B08 + B04)` per acquisition |
| group samples by `date.getMonth()` and average | `aggregate_temporal_period(period="month", reducer="mean")` |
| `stretch(avg, 0.1, 0.7)` | per-value linear stretch + clip to `[0, 1]` |
| `return [avg1, avg2, avg3]` (month → R/G/B) | `apply_dimension(t → bands)` + `rename_labels(["R", "G", "B"])` |
| (no cloud handling) | Scene Classification Layer (SCL) mask + `eo:cloud_cover` filter |

## License

This notebook is a conversion of a Sentinel Hub evalscript and is licensed under
**CC-BY-SA-4.0**.

Original evalscript: https://custom-scripts.sentinel-hub.com/custom-scripts/sentinel-2/agriculture_growth_stage/
Source: Sentinel Hub Custom Scripts (CC-BY-SA-4.0)
Conversion: Development Seed (openEO-UDP project)

## Import Required Libraries

We begin by importing the necessary Python libraries for data processing and
visualization.

In [ ]:
import datetime

import rioxarray
import matplotlib.pyplot as plt
from openeo.processes import clip

# openEO UDP parameter management system
from openeo_udp import ParameterManager

## Load Parameters and Connect to openEO Backend

Load algorithm parameters from the co-located parameter file and connect to an
openEO backend. We use the **Copernicus Data Space Ecosystem (CDSE)** because the
composite relies on the `aggregate_temporal_period` monthly reducer over every
Sentinel-2 acquisition of the three-month window.

In [ ]:
# Initialize the algorithm ID for UDP registration
_algorithm_id = "agriculture_growth_stage"

# Initialize parameter manager
param_manager = ParameterManager('agriculture_growth_stage.params.py')

# Display available options using the built-in helper
param_manager.print_options("agricultural growth stage algorithm")

In [ ]:
# Connect using a parameter set for a specified location on the CDSE endpoint.
# CDSE is required here because the monthly temporal aggregation needs the full
# Sentinel-2 archive for the window.
connection, current_params = param_manager.quick_connect(
    param_set="po_valley_cropland_italy",
    endpoint="ds_development",
)

## Load Sentinel-2 Data

Load Sentinel-2 L2A (atmospherically corrected) data across the three-month window.
We need:

- **B04** (665 nm, Red) and **B08** (833 nm, NIR): the two bands of NDVI
- **SCL**: Scene Classification Layer, used for cloud/shadow masking

`time` and `bounding_box` are passed as **`Parameter` objects** so the exported
process graph keeps `{"from_parameter": ...}` references and stays reusable as a
UDP. The scene-level `eo:cloud_cover` threshold is baked from the selected set.

In [ ]:
# Load Sentinel-2 data across the three-month window.
# Runtime knobs (time, bounding_box) are wired as Parameter refs; collection and
# bands are intrinsic to the algorithm and passed as concrete defaults.
s2cube = connection.load_collection(
    current_params["collection"].default,
    temporal_extent=current_params["time"],
    spatial_extent=current_params["bounding_box"],
    bands=current_params["bands"].default,
    properties={
        "eo:cloud_cover": lambda x: x <= current_params["cloud_cover"].default,
    },
)

print("✅ Sentinel-2 data loaded successfully!")

## Cloud and Shadow Masking (SCL)

The original `ORBIT` evalscript performs **no** cloud masking – it simply averages
every NDVI sample of each month, so residual clouds bias the monthly means. In
openEO we add the idiomatic **Scene Classification Layer (SCL)** mask, dropping
pixels classified as cloud shadow (3), cloud medium probability (8), cloud high
probability (9), thin cirrus (10) and snow/ice (11). This yields cleaner monthly
NDVI averages than the original.

In [ ]:
# Build a mask from the SCL band: True where we want to discard the pixel.
# Resolve the SCL band by prefix; the exact name varies by backend (e.g. "SCL"
# on CDSE, "SCL_20m" elsewhere).
scl_band = next(b for b in s2cube.metadata.band_names if b.lower().startswith("scl"))
scl = s2cube.band(scl_band)
invalid = (scl == 3) | (scl == 8) | (scl == 9) | (scl == 10) | (scl == 11)

# mask() removes pixels where the mask is non-zero (True).
s2_masked = s2cube.mask(invalid)

## NDVI per Acquisition

Reproduce the original `calcNDVI(sample)` for every acquisition in the time
series:

$$\text{NDVI} = \frac{B08 - B04}{B08 + B04}$$

NDVI is a normalized ratio, so the `reflectance_scale` divisor cancels out; we
still apply it (as in the sibling notebooks) so the same scaling conventions hold
across backends. The result is a single-band NDVI cube that still carries the full
temporal dimension.

In [ ]:
# NDVI = (B08 - B04) / (B08 + B04), built with the normalized_difference process.
# Resolve band names by prefix since backends may rename them (e.g. "B08_10m").
nir_band = next(b for b in s2_masked.metadata.band_names if b.lower().startswith("b08"))
red_band = next(b for b in s2_masked.metadata.band_names if b.lower().startswith("b04"))

scale = current_params["reflectance_scale"]  # cancels in NDVI; kept for consistency
nir = s2_masked.band(nir_band) / scale
red = s2_masked.band(red_band) / scale
ndvi = nir.normalized_difference(red)

## Monthly Mean NDVI

The original groups samples by `date.getMonth()` and averages them, producing
one mean NDVI per calendar month. CDSE on this federation node does not expose
`aggregate_temporal_period`, so we build explicit monthly intervals from the
`time` window and use **`aggregate_temporal`** with a `mean` reducer – the same
approach as the NDVI anomaly notebook. For a clean three-month window this
yields exactly **three** labels in chronological order (oldest → newest).

In [ ]:
# CDSE does not expose `aggregate_temporal_period`, so we build explicit
# monthly intervals from the (build-time) `time` window and use
# `aggregate_temporal`. Like the NDVI anomaly notebook, the intervals are
# computed in Python and baked into the graph; `bounding_box` and `cloud_cover`
# remain the runtime knobs of the exported UDP.
_start = datetime.date.fromisoformat(current_params["time"].default[0])
_end = datetime.date.fromisoformat(current_params["time"].default[1])


def _month_intervals(start, end):
    """List of [first_of_month, first_of_next_month] covering [start, end)."""
    intervals = []
    cur = start.replace(day=1)
    while cur < end:
        nxt = (cur.replace(day=28) + datetime.timedelta(days=4)).replace(day=1)
        intervals.append([cur.isoformat(), min(nxt, end).isoformat()])
        cur = nxt
    return intervals


month_intervals = _month_intervals(_start, _end)
month_labels = [a for a, _ in month_intervals]
assert len(month_intervals) == 3, f"expected 3 calendar months, got {len(month_intervals)}"

# Mean NDVI per calendar month -> a temporal dimension of length 3
# (oldest month first, most recent month last).
monthly_ndvi = ndvi.aggregate_temporal(
    intervals=month_intervals,
    labels=month_labels,
    reducer="mean",
)

## Stretch and Assign Months to R/G/B

The original applies `stretch(avg, 0.1, 0.7)` to each monthly mean – a linear
rescale of NDVI from `[0.1, 0.7]` to `[0, 1]` – and returns `[avg1, avg2, avg3]`
where `avg1` is the **oldest** month and `avg3` the **most recent**. In a single
`apply_dimension` step we stretch (and clip) every monthly mean and move the
three monthly time steps onto the **bands** dimension, giving a 3-band cube
ordered oldest → middle → newest, i.e. **R, G, B**.

Dimension names are read from `current_params` rather than hardcoded, since
backends differ (`t`/`time`, `bands`/`spectral`).

In [ ]:
# NDVI stretch range from the original `stretch(val, 0.1, 0.7)`.
STRETCH_MIN, STRETCH_MAX = 0.1, 0.7

t_dim = current_params["time_dimension"]
b_dim = current_params["bands_dimension"]


# Stretch each monthly NDVI mean to [0, 1] (clipped) and, in the same step, move
# the 3 monthly time steps onto the bands dimension. The callback receives the
# array of monthly means along time and returns a same-length array, so the
# source (time) dimension is replaced by a 3-element bands dimension.
#
# The 3 output bands are, in order: R = oldest month, G = middle, B = most
# recent. We deliberately do NOT rename the band labels: band *order* carries
# the R/G/B meaning in the exported GeoTIFF, and some backends (e.g. CDSE) do
# not expose the `rename_labels` process.
def stretch_to_bands(monthly_means):
    return monthly_means.array_apply(
        lambda v: clip((v - STRETCH_MIN) / (STRETCH_MAX - STRETCH_MIN), 0.0, 1.0)
    )


rgb = monthly_ndvi.apply_dimension(
    dimension=t_dim,
    target_dimension=b_dim,
    process=stretch_to_bands,
)

## Apply and Export

Save the 3-band growth-stage composite as a GeoTIFF.

In [ ]:
growth_stage_result = rgb.save_result("GTiff")

## Download and Visualize Results

Download the composite for the selected area and display it as an RGB image where
the red, green and blue channels are the oldest, middle and most recent monthly
NDVI means.

In [ ]:
# Synchronous execution (POST /result) does not accept unresolved Parameter refs,
# so materialize them with the current parameter set's defaults before download.
# The underlying parameterized graph is preserved for the UDP export cell below.
filename = f"{_algorithm_id}_{current_params['location_name'].replace(' ', '_').replace(',', '').lower()}.tif"

resolved = param_manager.resolve(growth_stage_result, current_params)
resolved.download(filename)

In [ ]:
# Open the 3-band result and display it as an RGB image.
ds = rioxarray.open_rasterio(filename)

# Move the band axis last and clip to [0, 1] for display.
rgb_img = ds.transpose("y", "x", "band").values
rgb_img = rgb_img.clip(0, 1)

fig, ax = plt.subplots(figsize=(12, 10))
ax.imshow(
    rgb_img,
    extent=[ds.x.values.min(), ds.x.values.max(), ds.y.values.min(), ds.y.values.max()],
)
_t = current_params["time"].default
ax.set_title(
    f"Agricultural Growth Stage (R/G/B = month 1/2/3)\n"
    f"{current_params['location_name']} | {_t[0]} → {_t[1]}",
    fontsize=12,
)
ax.axis("off")
plt.tight_layout()
plt.show()

## Interpretation Guide and Limitations

### Reading the result

Each pixel's colour encodes **when** its vegetation was greenest across the
three-month window (R = oldest month, G = middle, B = newest):

- **Red / Green / Blue** – a single sharp NDVI peak in the corresponding month.
- **Yellow (R+G)** or **Cyan (G+B)** – a long, even growing season spanning two
  consecutive months.
- **Purple / Magenta (R+B)** – a double peak with a drier middle month (e.g. two
  cropping cycles, or a cut-and-regrow forage).
- **White** – consistently high NDVI across all three months (dense, stable
  vegetation such as forest).
- **Dark** – consistently low NDVI (bare soil, water, urban).

### Limitations and differences from the original

- **SCL masking added.** The original `ORBIT` script averages every sample with no
  cloud handling; we mask clouds/shadows/snow via SCL, giving cleaner monthly
  means. Heavily clouded months may end up with few valid samples.
- **Calendar months, not rolling months.** `aggregate_temporal_period("month")`
  groups by calendar month, matching the original's `date.getMonth()` grouping.
  Choose a `time` window aligned to month boundaries (e.g. `2024-04-01` →
  `2024-07-01`) so exactly three months are produced; a partial month would add a
  fourth label and the `R/G/B` mapping would shift.
- **Stretch range.** The fixed `[0.1, 0.7]` stretch from the original is kept; it
  suits temperate cropland but can be widened for very dense or very sparse
  vegetation.

In [ ]:
# Create a directory to export images, UDP, and OGC API records
from pathlib import Path

_repo_root = next(p for p in Path.cwd().parents if (p / "notebooks").exists())
_alg_dir = _repo_root / "algorithm_registration" / _algorithm_id
_records_dir = _alg_dir / "records"
_udp_dir = _alg_dir / "openeo_udp"

_records_dir.mkdir(parents=True, exist_ok=True)
_udp_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Export the process graph for reuse as a parameterized UDP
import json

process_graph = {
    "process_graph": growth_stage_result.flat_graph(),
    "parameters": [
        current_params["time"].to_dict(),
        current_params["bounding_box"].to_dict(),
        current_params["cloud_cover"].to_dict(),
    ],
    "id": _algorithm_id,
    "summary": "Multi-temporal NDVI growth-stage RGB composite from Sentinel-2 imagery using openEO.",
    "description": (
        "Builds an RGB composite from a three-month window of Sentinel-2 L2A imagery "
        "by computing a mean NDVI for each calendar month (after SCL cloud masking) "
        "and assigning the oldest, middle and most recent monthly means to the red, "
        "green and blue channels. Colour therefore encodes crop phenology / growth "
        "stage. Converted from the Sentinel Hub Agricultural Growth Stage evalscript "
        "by @HarelDan (quarterly adaptation by András Zlinszky)."
    ),
}

with open(f"{_udp_dir}/{_algorithm_id}.json", "w") as f:
    json.dump(process_graph, f, indent=2)

print(f"Process graph exported to {_udp_dir}/{_algorithm_id}.json")
print(f"Process ID: {_algorithm_id}")

In [ ]:
# Export necessary metadata to register the process graph in APEx Algorithm Catalogue
_nb_href = f"{(Path.cwd() / f'{_algorithm_id}.ipynb').relative_to(_repo_root)}"

metadata = {
    "id": _algorithm_id,
    "title": "Agricultural Growth Stage",
    "preview_title": "Agricultural Growth Stage",
    "description": (
        "Multi-temporal NDVI composite that assigns the mean NDVI of each of three "
        "calendar months to the red, green and blue channels, so colour encodes when "
        "in the season vegetation peaked and separates crops by growth stage."
    ),
    "keywords": ["NDVI", "Phenology", "Growth stage", "Agriculture", "Multi-temporal", "Sentinel-2"],
    "themes": ["VEGETATION", "AGRICULTURE", "REMOTE SENSING", "Sentinel-2 MSI"],
    "created": "2026-06-25T00:00:00Z",
    "updated": "2026-06-25T00:00:00Z",
    "license": "CC-BY-SA-4.0",
    "openeo_backend_title": "CDSE openEO Federation",
    "openeo_backend_url": "https://openeofed.dataspace.copernicus.eu",
    "notebook_github_location": _nb_href,
    "collection_id": "SENTINEL2_L2A",
    "attribution": {
        "original_script": "https://custom-scripts.sentinel-hub.com/custom-scripts/sentinel-2/agriculture_growth_stage/",
        "authors": ["@HarelDan", "András Zlinszky"],
        "source_repository": "https://github.com/sentinel-hub/custom-scripts",
        "citation": None,
    },
}

with open(f"{_records_dir}/{_algorithm_id}.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Catalogue record exported to {_records_dir}/{_algorithm_id}.json")

## References and Attribution

**Original Script:** [Agricultural Growth Stage](https://custom-scripts.sentinel-hub.com/custom-scripts/sentinel-2/agriculture_growth_stage/)

**Authors:** @HarelDan (original), András Zlinszky (quarterly-mosaic adaptation)

**Source Repository:** [Sentinel Hub Custom Scripts](https://github.com/sentinel-hub/custom-scripts)

### openEO Conversion
- **Conversion Date**: 25 June 2026
- **openEO Framework**: Adapted for the openEO API and process graph structure
- **Backend Tested**: CDSE (Copernicus Data Space Ecosystem)
- **License**: CC-BY-SA-4.0

## Conclusion

This notebook converts the Agricultural Growth Stage algorithm into an openEO
User-Defined Process. The implementation:

✅ **Reproduces the multi-temporal NDVI logic** of the original `ORBIT` mosaicking
and per-month averaging using `aggregate_temporal_period("month", "mean")`.

✅ **Preserves the `[0.1, 0.7]` stretch** and the month → R/G/B channel mapping via
a single `apply_dimension` step (time steps → R/G/B bands).

✅ **Adds idiomatic openEO cloud masking** via the Scene Classification Layer,
yielding cleaner monthly means than the original.

✅ **Follows openEO standards** with an exported, parameterized process graph and
catalogue metadata.

### Key conversion notes
- `time`, `bounding_box` and `cloud_cover` are exposed as runtime UDP parameters;
  `collection` and `bands` are intrinsic to the algorithm.
- The `time` window should be aligned to whole calendar months so exactly three
  monthly labels (R, G, B) are produced.